# 00 — Setup & Dataset Download

This notebook downloads and prepares ISIC 2018 and ISIC 2020.
It creates stratified train/val/test CSVs.
Run it once before any training or EDA.

**Prerequisites:**
- Kaggle token configured (`~/.kaggle/access_token` or `~/.kaggle/kaggle.json`)
- `pip install kaggle scikit-learn pandas` (already in `environment.yml`)


In [1]:
import os
from pathlib import Path

# Ensure the working directory is the project root
root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"Working directory: {Path.cwd()}")


Working directory: /Users/davide/Desktop/Universita/AI/DL/project


## 1. Check Kaggle Authentication


In [2]:
import subprocess
from pathlib import Path

home = Path.home()
token_new = home / ".kaggle" / "access_token"
token_old = home / ".kaggle" / "kaggle.json"

if token_new.exists():
    print(f"[OK] Token found: {token_new}")
    perms = oct(token_new.stat().st_mode)[-3:]
    if perms != "600":
        print(f"[WARN] Permissions are {perms} instead of 600. Fix:")
        print("       chmod 600 ~/.kaggle/access_token")
elif token_old.exists():
    print(f"[OK] Token found: {token_old}")
    perms = oct(token_old.stat().st_mode)[-3:]
    if perms != "600":
        print(f"[WARN] Permissions are {perms} instead of 600. Fix:")
        print("       chmod 600 ~/.kaggle/kaggle.json")
else:
    raise EnvironmentError(
        "Kaggle token not found!\n"
        "Option A (new token): create ~/.kaggle/access_token\n"
        "  mkdir -p ~/.kaggle && echo YOUR_TOKEN > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token\n"
        "Option B (legacy token): download kaggle.json from Kaggle → Settings → API\n"
        "  mv ~/Downloads/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json"
    )

# Quick check: list datasets (does not download anything)
result = subprocess.run(["kaggle", "datasets", "list", "--max-size", "1"], capture_output=True, text=True)
if result.returncode == 0:
    print("[OK] Kaggle authentication works.")
else:
    print("[FAIL] Kaggle authentication failed:")
    print(result.stderr)
    raise EnvironmentError("Kaggle auth failed. Check your token.")


[OK] Token found: /Users/davide/.kaggle/access_token
[OK] Kaggle authentication works.


## 2. Prepare ISIC 2018


In [3]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "data/scripts/prepare_isic2018.py"],
    capture_output=False,
    text=True
)
if result.returncode != 0:
    raise RuntimeError("prepare_isic2018.py failed. Check the messages above.")


[INFO] ISIC 2018 already present (10015 images). Skipping download.
[INFO] Reading CSV from: data/raw/isic2018/GroundTruth.csv
[INFO] Total samples: 10015
[INFO] Train: 8012 (890 malignant, 11.1%)
[INFO] Val:   1001 (111 malignant, 11.1%)
[INFO] Test:  1002 (112 malignant, 11.2%)
[OK] CSV files saved to data/processed/isic2018


## 3. Prepare ISIC 2020


In [4]:
result = subprocess.run(
    [sys.executable, "data/scripts/prepare_isic2020.py"],
    capture_output=False,
    text=True
)
if result.returncode != 0:
    raise RuntimeError("prepare_isic2020.py failed. Check the messages above.")


[INFO] ISIC 2020 already present (33126 images). Skipping download.
[INFO] Images folder: data/raw/isic2020/train-image/image
[INFO] Metadata CSV:  data/raw/isic2020/train-metadata.csv
[INFO] CSV loaded: 33126 rows, columns: ['Unnamed: 0', 'isic_id', 'patient_id', 'target']
[INFO] ID column: 'isic_id' | Target column: 'target'
[INFO] Train: 26500 (467 malignant, 1.76%)
[INFO] Val:   3313 (59 malignant, 1.78%)
[INFO] Test:  3313 (58 malignant, 1.75%)
[OK]  CSV files saved to data/processed/isic2020


## 4. Final Check


In [5]:
import pandas as pd
from pathlib import Path

files = {
    "ISIC 2018 train": Path("data/processed/isic2018/train.csv"),
    "ISIC 2018 val": Path("data/processed/isic2018/val.csv"),
    "ISIC 2020 train": Path("data/processed/isic2020/train.csv"),
    "ISIC 2020 val": Path("data/processed/isic2020/val.csv"),
}

all_ok = True
for name, path in files.items():
    if not path.exists():
        print(f"[FAIL] {name}: {path} NOT FOUND")
        all_ok = False
        continue
    df = pd.read_csv(path)
    n_mal = df["target"].sum()
    print(f"[OK] {name:20s}: {len(df):6d} rows | {n_mal:5d} malignant ({n_mal/len(df):.2%})")

if all_ok:
    print("\nSetup complete. You can proceed with 01_eda.ipynb.")
else:
    print("\nSome files are missing. Check the errors above.")


[OK] ISIC 2018 train     :   8012 rows |   890 malignant (11.11%)
[OK] ISIC 2018 val       :   1001 rows |   111 malignant (11.09%)
[OK] ISIC 2020 train     :  26500 rows |   467 malignant (1.76%)
[OK] ISIC 2020 val       :   3313 rows |    59 malignant (1.78%)

Setup complete. You can proceed with 01_eda.ipynb.


## 5. Data Preview


In [6]:
from IPython.display import display

print("=== ISIC 2018 train (first 3 rows) ===")
df18 = pd.read_csv("data/processed/isic2018/train.csv")
display(df18.head(3))

print("\n=== ISIC 2020 train (first 3 rows) ===")
df20 = pd.read_csv("data/processed/isic2020/train.csv")
display(df20.head(3))


=== ISIC 2018 train (first 3 rows) ===


,image_id,filepath,target,source
0,ISIC_0033584,data/raw/isic2018/images/ISIC_0033584.jpg,0,isic2018
1,ISIC_0027388,data/raw/isic2018/images/ISIC_0027388.jpg,0,isic2018
2,ISIC_0030469,data/raw/isic2018/images/ISIC_0030469.jpg,0,isic2018



=== ISIC 2020 train (first 3 rows) ===


,image_id,filepath,target,source
0,ISIC_1429439,data/raw/isic2020/train-image/image/ISIC_14294...,0,isic2020
1,ISIC_0658963,data/raw/isic2020/train-image/image/ISIC_06589...,0,isic2020
2,ISIC_7459410,data/raw/isic2020/train-image/image/ISIC_74594...,0,isic2020


In [7]:
# Spot-check: verify that filepaths exist on disk
from pathlib import Path

for name, df in [("ISIC 2018", df18), ("ISIC 2020", df20)]:
    sample = df["filepath"].sample(5, random_state=0).tolist()
    missing = [p for p in sample if not Path(p).exists()]
    if missing:
        print(f"[FAIL] {name}: {len(missing)} filepaths do not exist:")
        for m in missing:
            print(f"  {m}")
    else:
        print(f"[OK] {name}: all sampled filepaths exist.")


[OK] ISIC 2018: all sampled filepaths exist.
[OK] ISIC 2020: all sampled filepaths exist.
